# End-to-end demo

Runs the full pipeline from loaded data through to liquidity sensitivity using the `ppa_exposure` package. This is the production-style usage; the step-by-step exploratory notebooks (01–07) show the development of each component.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the package importable from the notebooks/ directory
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ppa_exposure import (
    compute_exposure_metrics,
    compute_liquidity_metrics,
    compute_mtm_matrix,
    compute_par_fixed_price,
    simulate,
    threshold_sensitivity,
)
from ppa_exposure.validation import check_martingale, check_mtm_par_at_inception

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

## 1. Load data and calibrated parameters

In [ ]:
DATA = ROOT / "data"

# Forward curve
fc = pd.read_csv(next(DATA.glob("monthly_forward_curve_*.csv")))
fc["delivery_month"] = pd.to_datetime(fc["delivery_month"], format="%Y-%m")
forward_curve = fc["forward_eur_mwh"].values
delivery_months = pd.DatetimeIndex(fc["delivery_month"].values)

# Calibrated parameters
params = json.loads(next(DATA.glob("model_params*.json")).read_text())
kappa = params["parameters"]["kappa_per_year"]
sigma = params["parameters"]["sigma_annualized"]
shift_c = params["shift_constant_c"]

print(f"Forward curve: {len(forward_curve)} months, {delivery_months[0].strftime('%b-%y')} -> {delivery_months[-1].strftime('%b-%y')}")
print(f"Model: kappa={kappa:.2f} 1/year, sigma={sigma:.4f}, c={shift_c}")
print(f"Half-life: {np.log(2)/kappa*365.25:.1f} days")

## 2. Run Monte Carlo simulation

In [ ]:
valuation_date = (delivery_months[0] - pd.DateOffset(months=1)).replace(day=1)

sim = simulate(
    valuation_date=valuation_date,
    delivery_months=delivery_months,
    forward_curve=forward_curve,
    kappa=kappa, sigma=sigma, shift_constant_c=shift_c,
    n_paths=10000, use_antithetic=True, seed=42,
)

print(f"Simulated {sim.n_paths} paths over {len(sim.T_grid)} months.")

# Martingale validation
mart = check_martingale(sim)
print(f"Martingale check: max |z| = {mart.attrs['max_abs_z']:.2f}, passes = {mart.attrs['passes']}")

## 3. Compute par fixed price, MtM and exposure metrics

In [ ]:
F_fix = compute_par_fixed_price(forward_curve, delivery_months, sim.T_grid, discount_rate=0.02)
MtM = compute_mtm_matrix(sim, fixed_price=F_fix, volume_mw=100.0,
                          kappa=kappa, sigma=sigma, shift_constant_c=shift_c, discount_rate=0.02)
exposure = compute_exposure_metrics(MtM, delivery_months, F_fix, 100.0, discount_rate=0.02, pfe_quantile=0.95)

par_check = check_mtm_par_at_inception(MtM)
print(f"Par condition check: max |z-mean MtM| = {par_check['max_abs_z']:.2f}, passes = {par_check['passes']}")
print()
print(f"F_fix (par)              = {F_fix:.2f} EUR/MWh")
print(f"Notional                 = {exposure.notional_eur/1e6:.1f} M EUR")
print(f"Credit:   EPE = {exposure.EPE_credit/1e6:.2f} M, Max PFE = {exposure.Max_PFE_credit/1e6:.2f} M @ {exposure.Max_PFE_credit_month.strftime('%b-%y')}")
print(f"Liquidity: EPE = {exposure.EPE_liq/1e6:.2f} M, Max PFE = {exposure.Max_PFE_liq/1e6:.2f} M @ {exposure.Max_PFE_liq_month.strftime('%b-%y')}")

## 4. Apply CSA overlay (liquidity metrics)

In [ ]:
T = 5_000_000.0
liquidity = compute_liquidity_metrics(MtM, delivery_months, threshold=T)

print(f"With CSA threshold T = {T/1e6:.1f} M EUR:")
print(f"  Peak collateral PFE 95%: {liquidity.peak_pfe95/1e6:.2f} M EUR")
print(f"  Peak collateral PFE 99%: {liquidity.peak_pfe99/1e6:.2f} M EUR")
print(f"  Max single margin call PFE 95%: {liquidity.max_call_pfe95/1e6:.2f} M EUR")
print(f"  Fraction of paths with no posting: {liquidity.fraction_no_posting*100:.1f}%")

## 5. Threshold sensitivity

In [ ]:
thresholds = np.array([0, 2.5, 5, 7.5, 10, 15, 20, 30]) * 1e6
sens = threshold_sensitivity(MtM, thresholds, delivery_months)
display(sens.assign(**{c: sens[c]/1e6 for c in sens.columns if 'eur' in c}).round(2))

## 6. Visualize exposure profile

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, ee, pfe, label, color in [
    (axes[0], exposure.EE_credit, exposure.PFE_credit, 'Credit', 'steelblue'),
    (axes[1], exposure.EE_liq, exposure.PFE_liq, 'Liquidity (gross)', 'crimson'),
]:
    ax.fill_between(delivery_months, 0, pfe/1e6, alpha=0.25, color=color, label='PFE 95%')
    ax.plot(delivery_months, ee/1e6, linewidth=2, color=color, label='EE')
    ax.set_title(f'{label} exposure profile')
    ax.set_ylabel('M EUR')
    ax.legend()
plt.tight_layout()
plt.show()